### Re-ranking

In [1]:
import os
import sys
import joblib

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

model_name = "ngcf"
dir = f"artifacts/{model_name}"
prefix = f"{model_name}_k_all"    
eval_df = joblib.load(os.path.join(dir, f"{prefix}_eval_df.pkl"))
user_dps_df = joblib.load(os.path.join(dir, f"user_dps_df.pkl"))
feature_engineer = joblib.load(os.path.join(dir, f"feature_engineer.pkl"))

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


### MMR

In [2]:
from post_processing.mmr import MMR

mmr_reranker = MMR(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
)


Seed set to 42


In [3]:
mmr_result_df = mmr_reranker.rerank(
    top_k=20,
    theta=0.5,
)
mmr_result_df.head()

Preparing input DataFrame for MMR...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [00:30<00:00, 68.14it/s]


Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[318, 50, 1333, 475, 5952, 1199, 47, 1270, 36,...","[318, 50, 1333, 2858, 5952, 150, 475, 593, 715...","[2058, 163, 2490, 45722, 1233, 110, 2959, 2571]"
1,78,"[50, 47, 1199, 150, 1333, 4993, 293, 475, 36, ...","[50, 47, 1333, 4995, 6539, 7153, 150, 2858, 10...","[5881, 37729, 44191, 4119, 6993, 8400, 50872]"
2,127,"[46530, 58047, 3499, 6305, 3552, 714, 59615, 6...","[46530, 6305, 33166, 671, 37211, 3552, 45672, ...","[45726, 6958]"
3,170,"[4973, 57147, 2712, 1288, 33794, 5091, 593, 45...","[4973, 1288, 57147, 33794, 4526, 3804, 2988, 2...","[4963, 1222, 3949, 4011, 2542, 8874, 44191, 45..."
4,175,"[296, 47, 4993, 5669, 293, 1333, 1270, 2858, 8...","[296, 1036, 50, 1270, 47, 1089, 8961, 1704, 28...","[1921, 5995, 1913, 1419, 4927, 50068, 7700, 17..."


In [4]:
from common.eval import Evaluator
evaluator = Evaluator()

mmr_reranked_score_df = evaluator.evaluate(mmr_result_df, K=5)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=10)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=20)
mmr_reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.114466,0.013471,0.041473,0.146956,0.024220,0.038275,0.175514,0.042870,0.035562
std,20797.975208,0.268027,0.049562,0.098781,0.264182,0.067024,0.071601,0.249033,0.085964,0.055719
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.000000,0.000000,0.000000,0.301030,0.020944,0.100000,0.318073,0.058824,0.050000
max,71534.000000,1.000000,1.000000,0.800000,1.000000,1.000000,0.600000,1.000000,1.000000,0.450000


In [5]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=mmr_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 850.90it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.080914,0.849384,0.087918,0.825430,0.460911
std,595.969798,0.041694,0.095730,0.072104,0.085083,0.043222
min,0.000000,0.000000,0.479025,0.000000,0.326617,0.276897
25%,515.750000,0.052137,0.794834,0.029691,0.779886,0.432242
50%,1031.500000,0.080308,0.866955,0.079370,0.841970,0.462339
75%,1547.250000,0.108720,0.925606,0.133922,0.887840,0.492214
max,2063.000000,0.285642,0.999891,0.407356,0.979012,0.591286


In [6]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(mmr_result_df, k=10)
ils_df.describe()

,user,ILS@10
count,2064.000000,2064.000000
mean,35564.367733,0.097524
std,20797.975208,0.028209
min,75.000000,0.032460
25%,17798.500000,0.078519
50%,35054.000000,0.093007
75%,53331.000000,0.111296
max,71534.000000,0.238690


### Adaptive MMR

In [3]:
user_gs_type_dict = joblib.load(os.path.join("artifacts", 'user_gs_type_dict.pkl'))
print(user_gs_type_dict.keys())

dict_keys(['encoded_generalist', 'encoded_specialist', 'generalist', 'specialist'])


In [4]:
from post_processing.adaptive_mmr import AdaptiveMMR

adaptive_mmr_reranker = AdaptiveMMR(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
    user_type=user_gs_type_dict,
)

Seed set to 42


In [19]:
adaptive_mmr_result_df = adaptive_mmr_reranker.rerank(
    top_k=20,
    theta_s=0.95,
    theta_g=0.5,
)
adaptive_mmr_result_df.head()

Preparing input DataFrame for adaptive MMR...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [00:31<00:00, 65.24it/s]

Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[1333, 475, 1199, 50, 4993, 3996, 293, 2571, 1...","[1333, 318, 475, 50, 6711, 2858, 2571, 593, 86...","[2058, 163, 2490, 45722, 1233, 110, 2959, 2571]"
1,78,"[50, 1333, 47, 7153, 4993, 6539, 1036, 110, 49...","[50, 1333, 47, 7153, 6539, 4993, 1036, 5952, 1...","[5881, 37729, 44191, 4119, 6993, 8400, 50872]"
2,127,"[40966, 34321, 26554, 46855, 36401, 25788, 49,...","[40966, 26554, 44788, 30812, 6305, 50794, 4199...","[45726, 6958]"
3,170,"[4973, 152, 2716, 1061, 2396, 5922, 5787, 6385...","[4973, 2396, 2716, 152, 1061, 5922, 5787, 3379...","[4963, 1222, 3949, 4011, 2542, 8874, 44191, 45..."
4,175,"[1270, 4993, 475, 6711, 1036, 1333, 2858, 3349...","[1270, 4993, 6711, 475, 1036, 2858, 33493, 133...","[1921, 5995, 1913, 1419, 4927, 50068, 7700, 17..."


In [20]:
from common.eval import Evaluator
evaluator = Evaluator()

adaptive_mmr_reranked_score_df = evaluator.evaluate(adaptive_mmr_result_df, K=5)
adaptive_mmr_reranked_score_df = evaluator.evaluate(adaptive_mmr_reranked_score_df, K=10)
adaptive_mmr_reranked_score_df = evaluator.evaluate(adaptive_mmr_reranked_score_df, K=20)
adaptive_mmr_reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.121282,0.014606,0.043992,0.150354,0.025480,0.040116,0.180045,0.045413,0.036410
std,20797.975208,0.278134,0.053465,0.102904,0.272209,0.067888,0.076072,0.255881,0.092274,0.057397
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.000000,0.000000,0.000000,0.301030,0.020833,0.100000,0.324821,0.061543,0.050000
max,71534.000000,1.000000,1.000000,0.600000,1.000000,1.000000,0.600000,1.000000,1.000000,0.450000


In [21]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=adaptive_mmr_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 836.35it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.083412,0.888006,0.083170,0.811511,0.466525
std,595.969798,0.044190,0.107886,0.072787,0.098606,0.046082
min,0.000000,0.000000,0.505910,0.000000,0.356601,0.311050
25%,515.750000,0.050846,0.814951,0.018856,0.762165,0.435249
50%,1031.500000,0.081940,0.935452,0.074081,0.833738,0.468794
75%,1547.250000,0.113818,0.974167,0.126393,0.885072,0.498909
max,2063.000000,0.281618,0.999613,0.476192,0.968266,0.595666


### DPA-RS

In [7]:
from post_processing.dpa_rs import DPA_RS

reranker = DPA_RS(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
    ground_truth_dps_df=user_dps_df
)


Seed set to 42


In [8]:
dpa_result_df = reranker.rerank(
    top_k=20,
    max_iter=100,
    random_state=42,
)
dpa_result_df.head()

Preparing input DataFrame for DPA-RS...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064
Combined DataFrame shape: (206400, 16)
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [07:03<00:00,  4.87it/s]

Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[52281, 293, 8961, 6874, 1240, 150, 2571, 110,...","[318, 50, 1333, 2858, 5952, 150, 475, 593, 715...","[2058, 163, 2490, 45722, 1233, 110, 2959, 2571]"
1,78,"[441, 778, 48516, 541, 6377, 4973, 1240, 110, ...","[50, 47, 1333, 4995, 6539, 7153, 150, 2858, 10...","[5881, 37729, 44191, 4119, 6993, 8400, 50872]"
2,127,"[40732, 54775, 40412, 4532, 30848, 2001, 27839...","[46530, 6305, 33166, 671, 37211, 3552, 45672, ...","[45726, 6958]"
3,170,"[2406, 481, 1198, 3354, 2685, 4961, 7361, 1500...","[4973, 1288, 57147, 33794, 4526, 3804, 2988, 2...","[4963, 1222, 3949, 4011, 2542, 8874, 44191, 45..."
4,175,"[1704, 1391, 150, 2019, 48780, 2959, 2427, 192...","[296, 1036, 50, 1270, 47, 1089, 8961, 1704, 28...","[1921, 5995, 1913, 1419, 4927, 50068, 7700, 17..."


In [9]:
from common.eval import Evaluator
evaluator = Evaluator()

reranked_score_df = evaluator.evaluate(dpa_result_df, K=5)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=10)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=20)
reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.121905,0.012803,0.044477,0.155885,0.023558,0.040649,0.188567,0.043669,0.037209
std,20797.975208,0.275695,0.038445,0.101842,0.271991,0.054503,0.072687,0.255651,0.078373,0.055752
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.000000,0.000000,0.000000,0.315465,0.026757,0.100000,0.333333,0.062500,0.050000
max,71534.000000,1.000000,0.500000,0.800000,1.000000,0.500000,0.500000,1.000000,1.000000,0.500000


In [10]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=dpa_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 849.06it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.192069,0.975459,0.405191,0.921494,0.623553
std,595.969798,0.063369,0.026111,0.143264,0.035110,0.041673
min,0.000000,0.000000,0.687152,0.000000,0.703766,0.462525
25%,515.750000,0.152458,0.969712,0.322014,0.902924,0.598252
50%,1031.500000,0.192355,0.982560,0.414137,0.927115,0.626357
75%,1547.250000,0.231425,0.990596,0.493453,0.946617,0.650872
max,2063.000000,0.506708,1.000000,0.918092,0.993514,0.759521


In [11]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(dpa_result_df, k=10)
ils_df.describe()

,user,ILS@10
count,2064.000000,2064.000000
mean,35564.367733,0.272176
std,20797.975208,0.071672
min,75.000000,0.094321
25%,17798.500000,0.218974
50%,35054.000000,0.267062
75%,53331.000000,0.318704
max,71534.000000,0.538611
